# Anomaly detection 2/3 — Gaussian Mixture Models

TinyML course, Module 8. Adapted from the original course notebook (`Anomaly GMM.ipynb`), which follows Géron, *Hands-On ML*, ch. 9.

K-means scored by *distance* and drew spherical boundaries. A **GMM** models normal data as a mixture of K Gaussians — each with a mean **and a covariance matrix** — so clusters can be elongated and correlated. The anomaly score becomes a **log-likelihood**: samples in low-density regions are anomalies.

Plan:
1. Fix the elliptic case that broke K-means
2. `covariance_type` — the flexibility/parameter knob
3. Likelihood thresholding on **your fan features** and a head-to-head with K-means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

# The elliptic data that defeated K-means in notebook 1
X1, _ = make_blobs(n_samples=1000, centers=((4, -4), (0, 0)), random_state=42)
X1 = X1.dot(np.array([[0.374, 0.95], [0.732, 0.598]]))
X2, _ = make_blobs(n_samples=250, centers=1, random_state=42)
X2 = X2 + [6, -8]
X = np.r_[X1, X2]

gm = GaussianMixture(n_components=3, n_init=10, random_state=42).fit(X)
print('converged:', gm.converged_)
print('weights:', gm.weights_.round(3))

In [ ]:
def plot_gmm(gm, X, title=''):
    mins, maxs = X.min(0) - 0.5, X.max(0) + 0.5
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], 300),
                         np.linspace(mins[1], maxs[1], 300))
    Z = -gm.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, norm=LogNorm(vmin=1.0, vmax=30.0), levels=np.logspace(0, 2, 12))
    plt.contour(xx, yy, Z, norm=LogNorm(vmin=1.0, vmax=30.0), levels=np.logspace(0, 2, 12),
                linewidths=0.5, colors='k')
    plt.plot(X[:, 0], X[:, 1], 'k.', ms=2)
    plt.scatter(*gm.means_.T, marker='x', s=100, c='r')
    plt.title(title)

plt.figure(figsize=(8, 4))
plot_gmm(gm, X, 'GMM density contours: ellipses follow the data')
plt.show()

## `covariance_type` — how much shape freedom?

- `"spherical"` — circles of different radius (≈ K-means with soft assignments)
- `"diag"` — axis-aligned ellipses (covariance matrices are diagonal)
- `"tied"` — one shared full covariance for all clusters
- `"full"` (default) — every cluster its own full covariance

More freedom = more parameters = more data needed to fit them. On 13 fan features, a *full* covariance per component is 13×13 — with few clusters and enough windows that's fine; with many features consider `diag`.

In [ ]:
fig = plt.figure(figsize=(10, 7))
for i, ct in enumerate(['spherical', 'diag', 'tied', 'full']):
    gm_v = GaussianMixture(n_components=3, n_init=5, covariance_type=ct,
                           random_state=42).fit(X)
    plt.subplot(2, 2, i + 1)
    plot_gmm(gm_v, X, f'covariance_type="{ct}"')
plt.tight_layout(); plt.show()

## Anomaly detection = density thresholding

`gm.score_samples(X)` returns the log-likelihood of each sample. Pick a threshold such that a chosen fraction of training data falls below it — e.g. if you expect ~2 % outliers, use the 2nd percentile.

In [ ]:
densities = gm.score_samples(X)
density_threshold = np.percentile(densities, 2)
anomalies = X[densities < density_threshold]
print(f'density threshold: {density_threshold:.2f}')

plt.figure(figsize=(8, 4))
plot_gmm(gm, X, 'Low-density samples flagged as anomalies')
plt.scatter(anomalies[:, 0], anomalies[:, 1], color='r', marker='*', s=60)
plt.show()

## Part 2 — Your fan features

Same protocol as notebook 1 (so the numbers are directly comparable):
- fit on 70 % of the `normal` windows
- threshold = 2nd percentile of training log-likelihood
- score every other window, including the held-out fault class

Note the sign flip vs K-means: **low** likelihood = anomaly (we negate the score so that "bigger = more anomalous" like before).

In [ ]:
import os, sys
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sys.path.insert(0, os.path.abspath('../../module7-models/rf-features'))
from train_rf import load_windows   # noqa: E402

DATA_DIR = '../../module7-models/rf-features/data/raw'
HELD_OUT = 'scrape'   # keep identical to notebook 1

X_raw, X_feat, y, groups, _ = load_windows(DATA_DIR)

idx_normal = np.where(y == 'normal')[0]
rng = np.random.default_rng(0)
rng.shuffle(idx_normal)
train_idx = idx_normal[:int(0.7 * len(idx_normal))]

scaler = StandardScaler().fit(X_feat[train_idx])
Z = scaler.transform(X_feat)

gm_fan = GaussianMixture(n_components=3, n_init=10, covariance_type='full',
                         random_state=0).fit(Z[train_idx])
loglik = gm_fan.score_samples(Z)
thr_gmm = np.percentile(loglik[train_idx], 2)
print(f'GMM log-likelihood threshold: {thr_gmm:.2f}')

In [ ]:
# Head-to-head: GMM likelihood vs K-means distance on identical splits
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z[train_idx])
d = np.linalg.norm(Z[:, None, :] - km.cluster_centers_[None, :, :], axis=2).min(axis=1)
thr_km = np.percentile(d[train_idx], 98)

test_mask = np.ones(len(y), dtype=bool)
test_mask[train_idx] = False
df = pd.DataFrame({
    'state': y[test_mask],
    'gmm_anom': loglik[test_mask] < thr_gmm,
    'km_anom': d[test_mask] > thr_km,
})
summary = df.groupby('state').mean(numeric_only=True)
summary.columns = ['GMM flags', 'K-means flags']
print('Fraction of windows flagged per state (held-out =', HELD_OUT, '):')
print(summary.round(3))
summary.plot.bar(figsize=(8, 4), title='Detection rate per fan state')
plt.axhline(0.02, color='gray', ls=':', label='design false-alarm rate')
plt.legend(); plt.tight_layout(); plt.show()

**Discussion**

1. Where do GMM and K-means disagree? Plot the two scores against each other (scatter, coloured by state) to see *which* windows they disagree on.
2. Try `covariance_type='diag'` and 1–5 components. When does the extra flexibility of `full` stop paying?
3. On-device: a fitted GMM is just K means + K covariance matrices — **emlearn can convert `GaussianMixture` to C** (`emlearn.convert(gm)`), the same workflow as your Random Forest. Sketch what the firmware changes would be relative to Exercise 7.1. <!-- VERIFY: emlearn GMM/EllipticEnvelope support — listed under supported models in the emlearn docs -->

**Limitations:** both methods score *feature vectors*. If a fault only shows up as a waveform shape your 13 features can't see, neither detector can either.

**→ continue with `03_autoencoder_anomaly.ipynb`** — scoring raw windows, no feature engineering.